# 14 · PatchTST (Transformer)

Nie et al. (2023), *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers* (ICLR).

Implementação compacta, fiel ao código oficial (versão supervisionada):
- a janela é cortada em **patches** de 16 passos com passo 8 (o último valor é repetido 8 vezes no fim, como no original). Cada patch vira um "token";
- encoder Transformer com **BatchNorm** e **atenção residual** (os scores de uma camada somam aos da próxima), posição aprendida e cabeça linear "achatada";
- hiperparâmetros **do artigo, por base**: no ETTh1, `d_model=16`, 4 cabeças, 3 camadas, `d_ff=128`, dropout 0.3; no Weather, `d_model=128`, 16 cabeças, 3 camadas, `d_ff=256`, dropout 0.2;
- **sem RevIN interno**, porque o `ComRevIN` do protocolo já faz isso para todos.

Os patches têm 16 passos, então o teste `embaralhar_blocos` (blocos de 16) é especialmente informativo aqui.

In [1]:
# Configuração: igual em todos os notebooks
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJETO = '/content/drive/MyDrive/RN-Projeto'
except ImportError:          # fora do Colab (ex.: rodando no computador)
    PROJETO = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJETO, 'src'))
print('Pasta do projeto:', PROJETO)

import importlib
import rn_utils as ru
importlib.reload(ru)        # pega a versão mais nova se o 02 foi rodado de novo
ru.configurar(PROJETO)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/RN-Projeto
Projeto: /content/drive/MyDrive/RN-Projeto | dispositivo: cpu


## Modelo

`criar(L, H)` devolve um `nn.Module` que recebe `[lote, L, 1]` e devolve `[lote, H, 1]`. A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos os modelos.

In [2]:
import torch
import torch.nn as nn

class BatchNormTokens(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.bn = nn.BatchNorm1d(d)

    def forward(self, x):                                        # [B, N, d]
        return self.bn(x.transpose(1, 2)).transpose(1, 2)

class CamadaTST(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq, self.wk, self.wv = (nn.Linear(d_model, d_model) for _ in range(3))
        self.wo = nn.Sequential(nn.Linear(d_model, d_model), nn.Dropout(dropout))
        self.drop1, self.drop2 = nn.Dropout(dropout), nn.Dropout(dropout)
        self.norm1, self.norm2 = BatchNormTokens(d_model), BatchNormTokens(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x, scores_anteriores=None):
        B, N, d = x.shape
        q = self.wq(x).view(B, N, self.h, self.dk).transpose(1, 2)
        k = self.wk(x).view(B, N, self.h, self.dk).transpose(1, 2)
        v = self.wv(x).view(B, N, self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) * self.dk ** -0.5       # [B, h, N, N]
        if scores_anteriores is not None:                        # atenção residual
            scores = scores + scores_anteriores
        out = (torch.softmax(scores, dim=-1) @ v).transpose(1, 2).reshape(B, N, d)
        x = self.norm1(x + self.drop1(self.wo(out)))
        x = self.norm2(x + self.drop2(self.ff(x)))
        return x, scores

class PatchTST(nn.Module):
    def __init__(self, L, H, patch_len=16, stride=8, d_model=16, n_heads=4, n_layers=3, d_ff=128, dropout=0.3):
        super().__init__()
        self.patch_len, self.stride = patch_len, stride
        self.n_patches = (L - patch_len) // stride + 2
        self.pad = nn.ReplicationPad1d((0, stride))
        self.proj = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.empty(self.n_patches, d_model).uniform_(-0.02, 0.02))
        self.drop = nn.Dropout(dropout)
        self.camadas = nn.ModuleList([CamadaTST(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.cabeca = nn.Linear(d_model * self.n_patches, H)

    def forward(self, x):                                        # [B, L, 1] → [B, H, 1]
        z = self.pad(x.transpose(1, 2))                          # [B, 1, L + stride]
        z = z.unfold(-1, self.patch_len, self.stride)[:, 0]      # [B, N, patch_len]
        z = self.drop(self.proj(z) + self.pos)
        scores = None
        for camada in self.camadas:
            z, scores = camada(z, scores)
        return self.cabeca(z.flatten(1))[..., None]

CONFIG = {   # hiperparâmetros do artigo, por base
    'ETTh1':   dict(d_model=16,  n_heads=4,  n_layers=3, d_ff=128, dropout=0.3),
    'Weather': dict(d_model=128, n_heads=16, n_layers=3, d_ff=256, dropout=0.2),
}
def criar_etth1(L, H):
    return PatchTST(L, H, **CONFIG['ETTh1'])
def criar_weather(L, H):
    return PatchTST(L, H, **CONFIG['Weather'])
criar = criar_etth1   # usado no teste de formato

In [3]:
# Verificação do formato de entrada e saída
import torch
for L in (96, 720):
    m = criar(L, ru.HORIZONTE)
    saida = m(torch.randn(4, L, 1))
    assert saida.shape == (4, ru.HORIZONTE, 1), saida.shape
    print(f'L={L}: saída {tuple(saida.shape)} ✓ | {sum(p.numel() for p in m.parameters()):,} parâmetros')

L=96: saída (4, 96, 1) ✓ | 35,168 parâmetros
L=720: saída (4, 96, 1) ✓ | 156,224 parâmetros


## Treino e avaliação

O learning rate é escolhido pela validação (L = 336, seed 0) entre {1e-3, 3e-4, 1e-4}. Depois o modelo é treinado em todas as janelas e seeds das duas bases e avaliado no teste, com a janela original e com as quatro janelas embaralhadas. As execuções já salvas em `resultados/` são apenas recarregadas e listadas.

In [4]:
# hiperparâmetros do artigo para cada base
ru.rodar_grade('PatchTST', criar_etth1, datasets=['ETTh1'])
ru.rodar_grade('PatchTST', criar_weather, datasets=['Weather'])

lr escolhido para PatchTST em ETTh1: 0.0001  (val: {'0.001': 0.0942, '0.0003': 0.0933, '0.0001': 0.0933})
  · ETTh1_PatchTST_L96_H96_lr0.0001_s0 (salvo) | val 0.1007 | teste MSE 0.0555 MAE 0.1783 | embaralhar_tudo ×1.28 | 11 épocas, 19s (cuda)
  · ETTh1_PatchTST_L96_H96_lr0.0001_s1 (salvo) | val 0.1004 | teste MSE 0.0557 MAE 0.1785 | embaralhar_tudo ×1.28 | 13 épocas, 23s (cuda)
  · ETTh1_PatchTST_L96_H96_lr0.0001_s2 (salvo) | val 0.1003 | teste MSE 0.0556 MAE 0.1784 | embaralhar_tudo ×1.27 | 12 épocas, 21s (cuda)
  · ETTh1_PatchTST_L192_H96_lr0.0001_s0 (salvo) | val 0.0902 | teste MSE 0.0533 MAE 0.1756 | embaralhar_tudo ×1.43 | 30 épocas, 52s (cuda)
  · ETTh1_PatchTST_L192_H96_lr0.0001_s1 (salvo) | val 0.0924 | teste MSE 0.0545 MAE 0.1771 | embaralhar_tudo ×1.39 | 21 épocas, 38s (cuda)
  · ETTh1_PatchTST_L192_H96_lr0.0001_s2 (salvo) | val 0.0917 | teste MSE 0.0545 MAE 0.1774 | embaralhar_tudo ×1.41 | 21 épocas, 38s (cuda)
  · ETTh1_PatchTST_L336_H96_lr0.0001_s0 (salvo) | val 0.0933 | 

## Resumo (média das 3 seeds)

`×tudo`, `×antigo`, `×recente` e `×blocos` = MSE com a janela embaralhada / MSE original. ×1,00 significa que o modelo ignora aquela ordem.

In [5]:
df = ru.carregar_resultados()
r = df[df.modelo == 'PatchTST']
resumo = r.groupby(['dataset', 'L']).agg(**{
    'MSE val': ('mse_val', 'mean'), 'MSE teste': ('original_mse', 'mean'), 'dp': ('original_mse', 'std'),
    'MAE teste': ('original_mae', 'mean'), 'MSE sem suspeitos': ('original_mse_ss', 'mean'),
    **{'×' + p.replace('embaralhar_', ''): (f'{p}_razao', 'mean') for p in ru.PERTURBACOES},
    'épocas': ('epocas_rodadas', 'mean'), 'tempo (s)': ('tempo_treino_s', 'mean')})
resumo.round(4)

MSE val  MSE teste      dp  MAE teste  MSE sem suspeitos   ×tudo  \
dataset L                                                                       
ETTh1   96    0.1004     0.0556  0.0001     0.1784             0.0551  1.2787   
        192   0.0914     0.0541  0.0007     0.1767             0.0556  1.4130   
        336   0.0931     0.0542  0.0002     0.1786             0.0567  1.4560   
        512   0.0935     0.0576  0.0011     0.1867             0.0604  1.2847   
        720   0.0933     0.0585  0.0014     0.1897             0.0606  1.3224   
Weather 96    0.5406     0.4854  0.0012     0.4799             0.4854  1.2840   
        336   0.4468     0.5532  0.0207     0.5037             0.5532  1.1473   
        720   0.5083     0.4731  0.0649     0.4868             0.4731  1.3948   

             ×antigo  ×recente  ×blocos   épocas  tempo (s)  
dataset L                                                    
ETTh1   96    1.0347    1.1008   1.3602  12.0000    20.7620  
        192   1.0698    1.1621   1.5138  24.0000    42.9324  
        336   1.0438    1.2033   1.5797  12.3333    22.4205  
        512   0.9397    1.2026   1.4649   9.3333    16.8907  
        720   0.9542    1.2397   1.5102   9.3333    17.0058  
Weather 96    0.9819    1.1354   1.5831  16.0000   128.8514  
        336   0.8962    1.1622   3.3721  14.0000   119.4443  
        720   1.0312    1.2822   2.7336  10.6667   165.2160